# Day 45: Keyword search with BM25

**Build with AI: 100 Days to Job-Ready**, Chapter 13: Hybrid search and reranking

BM25 is the classic way to rank text by the words it shares with the question. It rewards rare words and short documents.

It is very good for exact things like form numbers and names, and blind to paraphrases.

**How to use:** run the cells from top to bottom (Runtime > Run all). No API key is needed.

> Needs: pip install rank-bm25. Uses chunks.json from Day 41.

In [ ]:
# Install (first run only)
!pip install -q rank-bm25

In [ ]:
# Setup: writes the small data files this lab reads into the Colab folder
import base64, pathlib
files = {
    'chunks.json': '[\r\n {\r\n  "id": 0,\r\n  "text": "Shifts run from 7:00 to 15:00 (morning) and 15:00 to 23:00 (evening). Staff must clock in at the counter tablet at least five minutes before the shift starts.",\r\n  "source": "handbook.html",\r\n  "where": "1. Working hours"\r\n },\r\n {\r\n  "id": 1,\r\n  "text": "A paid break of 30 minutes is given after four hours of work. Overtime beyond eight hours is paid at one and a half times the normal hourly rate.",\r\n  "source": "handbook.html",\r\n  "where": "1. Working hours"\r\n },\r\n {\r\n  "id": 2,\r\n  "text": "Every full-time employee gets 18 days of paid annual leave per year. Leave requests must be submitted on Form EX-7 at least seven days in advance.",\r\n  "source": "handbook.html",\r\n  "where": "2. Leave"\r\n },\r\n {\r\n  "id": 3,\r\n  "text": "Sick leave is 10 days per year. A doctor\'s note is needed if you are absent for more than two days in a row.",\r\n  "source": "handbook.html",\r\n  "where": "2. Leave"\r\n },\r\n {\r\n  "id": 4,\r\n  "text": "Marriage leave is 5 paid days and can be taken once. Parental leave is 12 weeks for the primary caregiver.",\r\n  "source": "handbook.html",\r\n  "where": "2. Leave"\r\n },\r\n {\r\n  "id": 5,\r\n  "text": "Staff can claim travel costs for work errands. Claims must be filed within 30 days and need the original bill.",\r\n  "source": "handbook.html",\r\n  "where": "3. Expenses"\r\n },\r\n {\r\n  "id": 6,\r\n  "text": "Meals during overtime are reimbursed up to Rs 200 per person per day. Taxi rides after 10 pm are reimbursed when the last bus has gone.",\r\n  "source": "handbook.html",\r\n  "where": "3. Expenses"\r\n },\r\n {\r\n  "id": 7,\r\n  "text": "Always wear closed shoes in the kitchen. Wet floors must be marked with a yellow sign right away.",\r\n  "source": "handbook.html",\r\n  "where": "4. Safety"\r\n },\r\n {\r\n  "id": 8,\r\n  "text": "In case of fire, leave through the back exit and gather near the parking lot. The shift manager takes a headcount.",\r\n  "source": "handbook.html",\r\n  "where": "4. Safety"\r\n },\r\n {\r\n  "id": 9,\r\n  "text": "The first-aid box is behind the counter. Burns must be cooled under running water for ten minutes before any cream is applied.",\r\n  "source": "handbook.html",\r\n  "where": "4. Safety"\r\n },\r\n {\r\n  "id": 10,\r\n  "text": "The Chai Corner apron and cap are provided and must be worn at the counter. Hair must be tied back and nails kept short and clean.",\r\n  "source": "handbook.html",\r\n  "where": "5. Dress code"\r\n },\r\n {\r\n  "id": 11,\r\n  "text": "Greet every customer within ten seconds. If a customer is unhappy with an order, replace it without argument and tell the shift manager.",\r\n  "source": "handbook.html",\r\n  "where": "6. Customers"\r\n },\r\n {\r\n  "id": 12,\r\n  "text": "Refunds up to Rs 500 can be approved by the shift manager. Larger refunds need the owner\'s approval.",\r\n  "source": "handbook.html",\r\n  "where": "6. Customers"\r\n },\r\n {\r\n  "id": 13,\r\n  "text": "Count the cash drawer at the start and end of every shift. Any difference above Rs 50 must be reported on the same day.",\r\n  "source": "handbook.html",\r\n  "where": "7. Cash handling"\r\n },\r\n {\r\n  "id": 14,\r\n  "text": "Never leave the drawer open when you walk away. Deposits to the bank are made every evening by the owner.",\r\n  "source": "handbook.html",\r\n  "where": "7. Cash handling"\r\n },\r\n {\r\n  "id": 15,\r\n  "text": "New staff complete a two-day training: day one covers hygiene and menu, day two covers billing and customers. A buddy from the team stays with you for the first week.",\r\n  "source": "handbook.html",\r\n  "where": "8. Training"\r\n },\r\n {\r\n  "id": 16,\r\n  "text": "Everyone must renew the food-safety certificate every year. The company pays the course fee.",\r\n  "source": "handbook.html",\r\n  "where": "8. Training"\r\n },\r\n {\r\n  "id": 17,\r\n  "text": "Staff get 30 percent off food and drinks when off duty. Friends and family receive 10 percent when you bring them in.",\r\n  "source": "handbook.html",\r\n  "where": "9. Staff discounts"\r\n },\r\n {\r\n  "id": 18,\r\n  "text": "For payroll questions write to payroll@chaicorner.example. For safety emergencies call the owner on the number in the back office.",\r\n  "source": "handbook.html",\r\n  "where": "10. Contact"\r\n },\r\n {\r\n  "id": 19,\r\n  "text": "closing The shop opens at 7:00.",\r\n  "source": "rules.pdf",\r\n  "where": "page 1"\r\n },\r\n {\r\n  "id": 20,\r\n  "text": "The first person in switches on the gas only after checking the pipe.",\r\n  "source": "rules.pdf",\r\n  "where": "page 1"\r\n },\r\n {\r\n  "id": 21,\r\n  "text": "Closing checklist: switch off the gas, clean the counter, count the cash drawer, lock the back door and set the alarm.",\r\n  "source": "rules.pdf",\r\n  "where": "page 1"\r\n },\r\n {\r\n  "id": 22,\r\n  "text": "The key to the back door stays with the shift manager at all times.",\r\n  "source": "rules.pdf",\r\n  "where": "page 1"\r\n },\r\n {\r\n  "id": 23,\r\n  "text": "Wash your hands with soap for 20 seconds before you start work and after every break.",\r\n  "source": "rules.pdf",\r\n  "where": "page 2"\r\n },\r\n {\r\n  "id": 24,\r\n  "text": "Milk must be kept below 5 degrees in the fridge.",\r\n  "source": "rules.pdf",\r\n  "where": "page 2"\r\n },\r\n {\r\n  "id": 25,\r\n  "text": "Throw away any milk that is more than two days old.",\r\n  "source": "rules.pdf",\r\n  "where": "page 2"\r\n },\r\n {\r\n  "id": 26,\r\n  "text": "Use separate cloths for tables and for the counter.",\r\n  "source": "rules.pdf",\r\n  "where": "page 2"\r\n },\r\n {\r\n  "id": 27,\r\n  "text": "Change the table cloth every two hours.",\r\n  "source": "rules.pdf",\r\n  "where": "page 2"\r\n },\r\n {\r\n  "id": 28,\r\n  "text": "We keep customer phone numbers only to send order updates. Numbers are deleted after 90 days.",\r\n  "source": "policy.txt",\r\n  "where": "paragraph"\r\n },\r\n {\r\n  "id": 29,\r\n  "text": "Staff photos are used only inside the shop. We never post a staff photo online without written consent.",\r\n  "source": "policy.txt",\r\n  "where": "paragraph"\r\n },\r\n {\r\n  "id": 30,\r\n  "text": "Surveillance cameras cover the counter and the entrance. They do not cover the staff rest room.",\r\n  "source": "policy.txt",\r\n  "where": "paragraph"\r\n },\r\n {\r\n  "id": 31,\r\n  "text": "A drink can be replaced or refunded if the customer says it is wrong within 15 minutes of the order. Food refunds need the receipt.",\r\n  "source": "policy.txt",\r\n  "where": "paragraph"\r\n },\r\n {\r\n  "id": 32,\r\n  "text": "Gift cards cannot be exchanged for cash. They are valid for one year from the date of purchase.",\r\n  "source": "policy.txt",\r\n  "where": "paragraph"\r\n },\r\n {\r\n  "id": 33,\r\n  "text": "Written complaints go to the owner within two working days. The owner replies within seven days.",\r\n  "source": "policy.txt",\r\n  "where": "paragraph"\r\n },\r\n {\r\n  "id": 34,\r\n  "text": "Anonymous complaints are accepted through the box near the exit and are read every Friday.",\r\n  "source": "policy.txt",\r\n  "where": "paragraph"\r\n }\r\n]',
}
for name, content in files.items():
    p = pathlib.Path(name)
    p.write_bytes(content if isinstance(content, bytes) else content.encode('utf-8'))
print('wrote', ', '.join(files))

## Lab: Where keywords win and fail

In [ ]:
import json, re
from rank_bm25 import BM25Okapi
chunks = json.load(open("chunks.json", encoding="utf-8"))
tok = lambda s: re.findall(r"[a-z0-9]+", s.lower())
bm25 = BM25Okapi([tok(c["text"]) for c in chunks])

def keyword_search(q, k=2):
    s = bm25.get_scores(tok(q))
    return [(round(float(s[i]), 1), chunks[i]["text"][:60]) for i in s.argsort()[::-1][:k]]

for q in ["Form EX-7", "How much time off do I get for my wedding?"]:
    print("Q:", q)
    for score, text in keyword_search(q):
        print("  ", score, text)

### Expected output (from the author's run; version numbers and timings may differ)

```
Q: Form EX-7
   7.1 Every full-time employee gets 18 days of paid annual leave p
   3.1 closing The shop opens at 7:00.
Q: How much time off do I get for my wedding?
   6.5 Staff get 30 percent off food and drinks when off duty. Frie
   3.4 Surveillance cameras cover the counter and the entrance. The
```

### Read it like this

- tok lowercases and splits the text into word pieces.
- get_scores gives one score per chunk.
- The exact code finds its chunk; the wedding question gets a weak match because 'wedding' is not in the handbook.

### Your turn

Try the query 'sick days'. Does BM25 find the sick leave paragraph? Why or why not?